Loading dataset

In [1]:
import json
import pandas as pd
import numpy as np
from collections import Counter

def squad2_to_df(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        squad = json.load(f)

    rows = []
    for article in squad["data"]:
        title = article["title"]
        for para_i, para in enumerate(article["paragraphs"]):
            context = para["context"]
            paragraph_id = f"{title}__{para_i}"

            for qa in para["qas"]:
                qid = qa["id"]
                question = qa["question"]
                is_impossible = qa.get("is_impossible", False)
                can_answer = "yes" if not is_impossible else "no"

                if can_answer == "yes" and len(qa.get("answers", [])) > 0:
                    answer_text  = qa["answers"][0]["text"]
                    answer_start = qa["answers"][0]["answer_start"]
                else:
                    answer_text  = "NO_ANSWER"
                    answer_start = -1

                rows.append({
                    "id":           qid,
                    "title":        title,
                    "paragraph_id": paragraph_id,
                    "context":      context,
                    "question":     question,
                    "can_answer":   can_answer,
                    "answer_text":  answer_text,
                    "answer_start": answer_start
                })

    return pd.DataFrame(rows)


def add_basic_features(df):
    df = df.copy()

    if "can_answer" in df.columns:
        df["y"] = (df["can_answer"].astype(str).str.lower() == "yes").astype(int)
    elif "is_impossible" in df.columns:
        df["y"] = (df["is_impossible"] == False).astype(int)
    else:
        df["y"] = (df["answer_text"] != "NO_ANSWER").astype(int)

    df["q_len_char"] = df["question"].astype(str).str.len()
    df["c_len_char"] = df["context"].astype(str).str.len()
    df["q_len_tok"]  = df["question"].astype(str).str.split().str.len()
    df["c_len_tok"]  = df["context"].astype(str).str.split().str.len()

    if "answer_text" in df.columns:
        df["a_len_char"] = df["answer_text"].astype(str).str.len()
        df["a_len_tok"]  = df["answer_text"].astype(str).str.split().str.len()
        df.loc[df["answer_text"] == "NO_ANSWER", ["a_len_char", "a_len_tok"]] = 0

    return df


train_df = squad2_to_df("train_sampled.json")
val_df   = squad2_to_df("val_sampled.json")
test_df  = squad2_to_df("test_sampled.json")

train_df = add_basic_features(train_df)
val_df   = add_basic_features(val_df)
test_df  = add_basic_features(test_df)

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

Train shape: (8001, 15)
Val shape:   (996, 15)
Test shape:  (1004, 15)


Upload del best model 

In [2]:
import torch
from transformers import DebertaV2ForQuestionAnswering

SAVE_PATH  = "deberta_squad_finetuned"
device = "mps" if torch.backends.mps.is_available() else "cpu"

best_model = DebertaV2ForQuestionAnswering.from_pretrained(SAVE_PATH)
best_model.to(device)
best_model.eval()

print("Best model loaded.")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

/opt/anaconda3/envs/tf/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 9291.05it/s]


Best model loaded.


In [4]:
pip install sentence_transformers

Note: you may need to restart the kernel to use updated packages.


In [11]:
# ── Blocco 1: Installazione ──────────────────────────────────
# !pip install sentence-transformers scikit-learn

# ── Blocco 2: Costruisci indice ibrido ──────────────────────
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import torch
import pandas as pd

# Tutti i contesti unici dal dataset
all_contexts = pd.concat([train_df, val_df, test_df])["context"].unique().tolist()
print(f"Contesti totali nell'indice: {len(all_contexts)}")

# Sentence embeddings con modello ottimizzato per QA
print("Caricando modello embedding...")
embedder = SentenceTransformer("multi-qa-mpnet-base-dot-v1")

print("Calcolando embeddings dei contesti...")
ctx_embeddings = embedder.encode(
    [f"passage: {ctx}" for ctx in all_contexts],
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Embeddings shape: {ctx_embeddings.shape}")

# TF-IDF
print("Costruendo indice TF-IDF...")
vectorizer   = TfidfVectorizer(max_features=50000, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(all_contexts)
print("Indice ibrido costruito.")

# ── Blocco 3: Retriever ibrido ───────────────────────────────
def retrieve(question, k=5, alpha=0.5):
    # Score semantica con dot product invece di cosine
    q_emb      = embedder.encode([f"query: {question}"], convert_to_numpy=True)
    sem_scores = np.dot(ctx_embeddings, q_emb.T).flatten()

    # Score TF-IDF
    q_tfidf    = vectorizer.transform([question])
    lex_scores = cosine_similarity(q_tfidf, tfidf_matrix).flatten()

    # Normalizza
    sem_scores = (sem_scores - sem_scores.min()) / (sem_scores.max() - sem_scores.min() + 1e-9)
    lex_scores = (lex_scores - lex_scores.min()) / (lex_scores.max() - lex_scores.min() + 1e-9)

    # Combina — dai più peso alla semantica
    combined = 0.7 * sem_scores + 0.3 * lex_scores
    top_k    = combined.argsort()[-k:][::-1]
    return [(all_contexts[i], float(combined[i])) for i in top_k]

# ── Blocco 4: Pipeline RAG completa ─────────────────────────
def rag(question, model, tokenizer, device, k=5, alpha=0.5, verbose=True):
    # Step 1: Retrieve
    retrieved = retrieve(question, k=k, alpha=alpha)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Domanda: {question}")
        print(f"{'='*60}")
        print(f"\n📚 Top {k} contesti recuperati:")
        for j, (ctx, score) in enumerate(retrieved):
            print(f"\n  [{j+1}] Score: {score:.4f}")
            print(f"  {ctx[:200]}...")

    # Step 2: Read — leggi tutti i contesti e prendi lo span migliore
    best_answer     = None
    best_context    = None
    best_span_score = -float("inf")

    model.eval()
    for ctx, retrieval_score in retrieved:
        inputs = tokenizer(
            question,
            ctx,
            max_length=384,
            truncation="only_second",
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
            return_tensors="pt"
        )

        offset_mapping = inputs.pop("offset_mapping")
        inputs.pop("overflow_to_sample_mapping")
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model(**inputs)

        for i in range(len(outputs.start_logits)):
            start_logits = outputs.start_logits[i]
            end_logits   = outputs.end_logits[i]
            offsets      = offset_mapping[i]

            null_score = start_logits[0] + end_logits[0]
            span_start = int(start_logits[1:].argmax()) + 1
            span_end   = int(end_logits[1:].argmax()) + 1
            span_score = start_logits[span_start] + end_logits[span_end]

            if span_score > null_score and span_score > best_span_score:
                best_span_score = span_score.item()
                if span_end < span_start:
                    span_end = span_start
                start_char   = offsets[span_start][0].item()
                end_char     = offsets[span_end][1].item()
                best_answer  = ctx[start_char:end_char]
                best_context = ctx

    if verbose:
        print(f"\n{'='*60}")
        if best_answer:
            print(f"✅ Risposta: {best_answer}")
            print(f"\n📄 Dal contesto:\n{best_context[:300]}...")
        else:
            print(f"❌ Il modello non sa rispondere a questa domanda.")
        print(f"{'='*60}\n")

    return best_answer

# ── Blocco 5: Test ───────────────────────────────────────────
# Scienza
rag("What is the theory of relativity?", best_model, tokenizer, device)
rag("Who invented the telephone?", best_model, tokenizer, device)
rag("What is the speed of light?", best_model, tokenizer, device)

# Storia
rag("Who was Napoleon Bonaparte?", best_model, tokenizer, device)
rag("When did the American Civil War start?", best_model, tokenizer, device)
rag("Who wrote the Declaration of Independence?", best_model, tokenizer, device)

# Geografia
rag("What is the longest river in the world?", best_model, tokenizer, device)
rag("What is the population of New York City?", best_model, tokenizer, device)

# Sport
rag("Who won the first Super Bowl?", best_model, tokenizer, device)
rag("When were the first Olympic Games held?", best_model, tokenizer, device)

# Domande senza risposta nel dataset
rag("Who is the CEO of Apple in 2024?", best_model, tokenizer, device)
rag("What is the price of Bitcoin today?", best_model, tokenizer, device)

Contesti totali nell'indice: 2140
Caricando modello embedding...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8553.66it/s]
MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Calcolando embeddings dei contesti...


Batches: 100%|██████████| 34/34 [00:45<00:00,  1.35s/it]


Embeddings shape: (2140, 768)
Costruendo indice TF-IDF...
Indice ibrido costruito.

Domanda: What is the theory of relativity?

📚 Top 5 contesti recuperati:

  [1] Score: 0.9571
  In the classical case, the invariance, or symmetry, group and the covariance group coincide, but, interestingly enough, they part ways in relativistic physics. The symmetry group of the general theory...

  [2] Score: 0.8270
  The beginning of the 20th century brought the start of a revolution in physics. The long-held theories of Newton were shown not to be correct in all circumstances. Beginning in 1900, Max Planck, Alber...

  [3] Score: 0.7937
  Idealist notions took a strong hold among physicists of the early 20th century confronted with the paradoxes of quantum physics and the theory of relativity. In The Grammar of Science, Preface to the ...

  [4] Score: 0.7178
  Important geological concepts were established as naturalists began studying the rock formations of the Alps in the 18th century. In the mi

In [10]:
# Test manuale: dai direttamente il contesto giusto a DeBERTa
context = """The telephone was invented by Alexander Graham Bell, 
who was awarded the first patent for the telephone in 1876. 
Bell's invention allowed people to communicate over long distances 
using electrical signals."""

question = "Who invented the telephone?"

answer = read(question, context, best_model, tokenizer, device)
print(f"Q: {question}")
print(f"A: {answer}")

Q: Who invented the telephone?
A:  Alexander Graham Bell
